# Debug — Calcul risque journalier
Visualisation pas-à-pas de la répartition IFT sur les périodes de traitement et du calcul de risque.

In [1]:
import sys
sys.path.insert(0, '../../etl')

import polars as pl
import pandas as pd
import duckdb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import date, timedelta
from calcul_risque_journalier import CULTURE_MAPPING, normaliser_culture

PARQUET = '../../data/parquet'
ANNEE   = 2023
INSEE   = '37031'   # Bourgueil — changer ici pour tester une autre commune
METHODE_SEUIL = "quartiles"  #Sinon, "valeurs"
VALEURS_SEUIL = 1,2,3

## 1. Données IFT de la commune

In [2]:
# Chargement des données de la commune
ift_raw = pl.read_parquet(f'{PARQUET}/ift_communes_enrichi.parquet')
commune = ift_raw.filter(pl.col('insee_com') == INSEE)
commune

id,insee_com,sau,sau_bio,p_bio,p_bc,p_sau,c_maj,c_ift_hbc,c_ift_h,cod_c_maj,cod_c_hbc,cod_c_h,ift_t,ift_t_hbc,ift_h,ift_t_hh,ift_hh_hbc,iftbc,c_maj_depuis_code,c_ift_hbc_depuis_code,c_ift_h_depuis_code,code_insee_dep,nom_commune,code_insee_dep_right,code_insee_reg,longitude,latitude
str,str,f64,f64,f64,f64,f64,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,f64,f64
"""COMMUNE_0000000009745772""","""37031""",843.09,208.21,24.7,19.05,25.58,"""Vigne""","""Vigne""","""Blé tendre""","""Vigne""","""Vigne""","""BTH""",5.09,4.12,0.95,4.14,3.17,0.97,null,null,"""Blé tendre d'hiver""","""37""","""BOURGUEIL""","""37""","""24""",0.174758,47.309531


In [3]:
# Mapping des cultures
CULTURE_MAPPING

{'Blé tendre': 'Blé tendre',
 'Blé dur': 'Blé dur',
 "Orge d'hiver": "Orge d'hiver",
 'Orge de printemps': 'Orge de printemps',
 'Triticale': "Céréales d'hiver",
 'Épeautre': "Céréales d'hiver",
 "Seigle d'hiver": "Céréales d'hiver",
 'Avoine': 'Céréales',
 'Seigle de printemps': 'Céréales',
 'Mélange de céréales': 'Céréales',
 'Maïs': 'Maïs',
 'Maïs ensilage': 'Maïs',
 'Maïs doux': 'Maïs',
 'Colza': 'Colza',
 'Tournesol': 'Tournesol',
 'Lin fibres': 'Lin printemps',
 'Lin oléagineux': 'Lin printemps',
 'Sorgho': 'Sorgho',
 'Millet': 'Millet',
 'Sarrasin': 'Céréales',
 'Riz': 'Céréales',
 'Betterave non fourragère / Bette': 'Betterave',
 'Pomme de terre de consommation': 'Pomme de terre de consommation',
 'Pomme de terre féculière': 'Pomme de terre de consommation',
 'Pomme de terre primeur': 'Pomme de terre de consommation',
 'Féverole': 'Féverole',
 'Pois protéagineux': 'Pois protéagineux',
 'Petits pois': 'Pois de printemps',
 'Lentille cultivée (non fourragère)': 'Pois de printemps

In [4]:
# Application du mapping
old, new = list(CULTURE_MAPPING.keys()), list(CULTURE_MAPPING.values())
commune = commune.with_columns([
    pl.col('c_maj').replace_strict(old=old, new=new, default=None).alias('c_maj_cal'),
    pl.col('c_ift_hbc').replace_strict(old=old, new=new, default=None).alias('c_ift_hbc_cal'),
    pl.col('c_ift_h').replace_strict(old=old, new=new, default=None).alias('c_ift_h_cal'),
])
commune

id,insee_com,sau,sau_bio,p_bio,p_bc,p_sau,c_maj,c_ift_hbc,c_ift_h,cod_c_maj,cod_c_hbc,cod_c_h,ift_t,ift_t_hbc,ift_h,ift_t_hh,ift_hh_hbc,iftbc,c_maj_depuis_code,c_ift_hbc_depuis_code,c_ift_h_depuis_code,code_insee_dep,nom_commune,code_insee_dep_right,code_insee_reg,longitude,latitude,c_maj_cal,c_ift_hbc_cal,c_ift_h_cal
str,str,f64,f64,f64,f64,f64,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,f64,f64,str,str,str
"""COMMUNE_0000000009745772""","""37031""",843.09,208.21,24.7,19.05,25.58,"""Vigne""","""Vigne""","""Blé tendre""","""Vigne""","""Vigne""","""BTH""",5.09,4.12,0.95,4.14,3.17,0.97,null,null,"""Blé tendre d'hiver""","""37""","""BOURGUEIL""","""37""","""24""",0.174758,47.309531,"""Vigne""","""Vigne""","""Blé tendre"""


In [5]:
# Description de la commune
r = commune.to_pandas().iloc[0]
dep = r['code_insee_dep']
print(f"Commune  : {r.get('nom_commune', INSEE)} ({INSEE}) — dép. {dep}")
print()
print(f"{'':40s} {'Culture IFT':30s}  {'→ Calendrier':25s}  {'IFT annuel':>10}")
print('-' * 110)
for col_raw, col_cal, ift_col, type_ttt in [
    ('c_maj',     'c_maj_cal',     'ift_t_hbc',    'Total (toutes périodes)'),
    ('c_ift_hbc', 'c_ift_hbc_cal', 'ift_hh_hbc', 'Total hors herbicides (toutes périodes)'),
    ('c_ift_h',   'c_ift_h_cal',   'ift_h',     'Herbicides uniquement'),
]:
    raw = r.get(col_raw, '—')
    cal = r.get(col_cal)
    ift_val = r.get(ift_col, 0) or 0
    cal_str = str(cal) if pd.notna(cal) else '⚠ HORS CALENDRIER'
    print(f"  {type_ttt:38s} {str(raw):30s}  {cal_str:25s}  {ift_val:>10.3f}")

Commune  : BOURGUEIL (37031) — dép. 37

                                         Culture IFT                     → Calendrier               IFT annuel
--------------------------------------------------------------------------------------------------------------
  Total (toutes périodes)                Vigne                           Vigne                           4.120
  Total hors herbicides (toutes périodes) Vigne                           Vigne                           3.170
  Herbicides uniquement                  Blé tendre                      Blé tendre                      0.950


## 2. Calendrier d'épandage pour cette commune (normalisé vers l'année cible)

In [6]:
# Recupération des données
con = duckdb.connect(':memory:')
cal_df = con.execute(f"""
    SELECT
        culture,
        make_date({ANNEE}, month(Debut_de_periode), day(Debut_de_periode)) AS debut,
        make_date({ANNEE}, month(Fin_de_periode),   day(Fin_de_periode))   AS fin,
        Herbicides, Fongicides, Insecticides,
        COUNT(*) FILTER (WHERE Herbicides)
            OVER (PARTITION BY culture) AS nb_periodes_herbicides,
        COUNT(*) FILTER (WHERE Fongicides OR Insecticides)
            OVER (PARTITION BY culture) AS nb_periodes_fongi_insecti,
        COUNT(*) OVER (PARTITION BY culture) AS nb_periodes_total
    FROM read_parquet('{PARQUET}/calendrier_epandage.parquet')
    WHERE departement_code = {int(dep)}
""").df()
con.close()

cal_df

,culture,debut,fin,Herbicides,Fongicides,Insecticides,nb_periodes_herbicides,nb_periodes_fongi_insecti,nb_periodes_total
0,Blé dur,2023-01-01,2023-02-20,True,False,False,2,2,4
1,Blé dur,2023-04-15,2023-04-30,True,True,False,2,2,4
2,Blé dur,2023-05-01,2023-06-14,False,True,False,2,2,4
3,Blé dur,2023-11-15,2023-12-31,False,False,False,2,2,4
4,Blé tendre,2023-02-01,2023-03-14,True,False,False,4,5,7
...,...,...,...,...,...,...,...,...,...
56,Tournesol,2023-12-15,2023-12-31,True,False,False,4,2,4
57,Vigne,2023-01-01,2023-03-30,True,False,False,3,2,4
58,Vigne,2023-04-01,2023-04-30,True,True,False,3,2,4
59,Vigne,2023-05-01,2023-08-15,False,True,True,3,2,4


In [7]:
# Filtrer sur les cultures de la commune
cultures_commune = {r.get(c) for c in ('c_maj_cal','c_ift_hbc_cal','c_ift_h_cal') if pd.notna(r.get(c))}
print(f"Cultures recherchées dans le calendrier : {cultures_commune}")
print()
cal_commune = cal_df[cal_df['culture'].isin(cultures_commune)].sort_values(['culture','debut'])
display(cal_commune)

Cultures recherchées dans le calendrier : {'Blé tendre', 'Vigne'}



,culture,debut,fin,Herbicides,Fongicides,Insecticides,nb_periodes_herbicides,nb_periodes_fongi_insecti,nb_periodes_total
4,Blé tendre,2023-02-01,2023-03-14,True,False,False,4,5,7
5,Blé tendre,2023-03-15,2023-04-14,False,True,False,4,5,7
6,Blé tendre,2023-04-15,2023-04-30,True,True,False,4,5,7
7,Blé tendre,2023-05-01,2023-06-14,False,True,True,4,5,7
8,Blé tendre,2023-10-01,2023-10-14,False,False,True,4,5,7
9,Blé tendre,2023-10-15,2023-11-30,True,False,True,4,5,7
10,Blé tendre,2023-12-01,2023-12-31,True,False,False,4,5,7
57,Vigne,2023-01-01,2023-03-30,True,False,False,3,2,4
58,Vigne,2023-04-01,2023-04-30,True,True,False,3,2,4
59,Vigne,2023-05-01,2023-08-15,False,True,True,3,2,4


## 3. IFT journalier — répartition sur les périodes

In [9]:
# Pour chaque couple (culture, type_traitement), calculer l'IFT par période
configs = [
    ('c_maj_cal',     'ift_t_hbc',   None,                                              'nb_periodes_total',         'Total (c_maj)'),
    ('c_ift_hbc_cal', 'ift_hh_hbc',  lambda df: df['Fongicides'] | df['Insecticides'],  'nb_periodes_fongi_insecti', 'Fongicides/Insecticides (c_ift_hbc)'),
    ('c_ift_h_cal',   'ift_h',       lambda df: df['Herbicides'],                       'nb_periodes_herbicides',    'Herbicides (c_ift_h)'),
]

# Flags de déduplication : c_ift_hbc et c_ift_h inclus seulement si différents de c_maj (noms calendrier)
c_maj_cal_val   = r.get('c_maj_cal')
c_ift_hbc_cal   = r.get('c_ift_hbc_cal')
c_ift_h_cal     = r.get('c_ift_h_cal')
hbc_different   = (c_ift_hbc_cal != c_maj_cal_val) if (pd.notna(c_ift_hbc_cal) or pd.notna(c_maj_cal_val)) else True
h_different     = (c_ift_h_cal   != c_maj_cal_val) if (pd.notna(c_ift_h_cal)   or pd.notna(c_maj_cal_val)) else True

print("── Déduplication ──────────────────────────────────────────")
print(f"  c_maj_cal     : {c_maj_cal_val}")
print(f"  c_ift_hbc_cal : {c_ift_hbc_cal}  {'→ INCLUS' if hbc_different else '→ EXCLU (même que c_maj)'}")
print(f"  c_ift_h_cal   : {c_ift_h_cal}  {'→ INCLUS' if h_different else '→ EXCLU (même que c_maj)'}")
print()

print(f"{'Config':42s} {'Culture cal':20s} {'IFT annuel':>10} {'Nb périodes':>12} {'IFT/période':>12} {'Inclus':>7}")
print('-' * 108)
for col_cal, ift_col, filtre, col_nb, label in configs:
    culture = r.get(col_cal)
    ift_val = r.get(ift_col, 0) or 0

    # Flag d'inclusion
    if col_cal == 'c_maj_cal':
        inclus = True
    elif col_cal == 'c_ift_hbc_cal':
        inclus = hbc_different
    else:
        inclus = h_different

    if pd.isna(culture):
        print(f"  {label:40s} {'⚠ hors calendrier':20s} {ift_val:>10.3f} {'—':>12} {'—':>12} {'—':>7}")
        continue

    periodes = cal_commune[cal_commune['culture'] == culture]
    if filtre is not None:
        periodes = periodes[filtre(periodes)]

    nb = periodes[col_nb].iloc[0] if not periodes.empty else 0
    ift_par_periode = ift_val / nb if nb > 0 else 0
    flag = '✓' if inclus else '✗ exclu'
    print(f"  {label:40s} {str(culture):20s} {ift_val:>10.3f} {nb:>12} {ift_par_periode:>12.4f} {flag:>7}")

── Déduplication ──────────────────────────────────────────
  c_maj_cal     : Vigne
  c_ift_hbc_cal : Vigne  → EXCLU (même que c_maj)
  c_ift_h_cal   : Blé tendre  → INCLUS

Config                                     Culture cal          IFT annuel  Nb périodes  IFT/période  Inclus
------------------------------------------------------------------------------------------------------------
  Total (c_maj)                            Vigne                     4.120            4       1.0300       ✓
  Fongicides/Insecticides (c_ift_hbc)      Vigne                     3.170            2       1.5850 ✗ exclu
  Herbicides (c_ift_h)                     Blé tendre                0.950            4       0.2375       ✓


## 4. Simulation du risque jour par jour sur l'année

In [10]:
from calcul_risque_journalier import compute_ift_journalier

In [11]:
DUCKDB_PATH = "../../data/pestiexpo.duckdb"
PARQUET_DIR = "../../data/parquet"
METEO_ENABLED = False

In [ ]:
def load_data(annee: int, METEO_ENABLED) -> tuple:

    con = duckdb.connect(str(DUCKDB_PATH))

    ift_raw = con.execute("""
        SELECT
            insee_com,
            code_insee_dep,
            c_maj,     ift_t_hbc  AS ift_maj_hbc,
            c_ift_hbc, ift_hh_hbc AS ift_hh_hbc,
            c_ift_h,   ift_h      AS ift_h
        FROM ift_communes_enrichi
        WHERE ift_t IS NOT NULL
    """).pl()

    # Normalisation des noms de culture vers les noms du calendrier
    ift = ift_raw.with_columns([
        normaliser_culture(pl.col("c_maj")).alias("c_maj_cal"),
        normaliser_culture(pl.col("c_ift_hbc")).alias("c_ift_hbc_cal"),
        normaliser_culture(pl.col("c_ift_h")).alias("c_ift_h_cal"),
    ])

    cal = con.execute(f"""
        SELECT
            departement_code,
            culture,
            make_date({annee}, month(Debut_de_periode), day(Debut_de_periode)) AS Debut_de_periode,
            make_date({annee}, month(Fin_de_periode),   day(Fin_de_periode))   AS Fin_de_periode,
            Herbicides,
            Fongicides,
            Insecticides,
            COUNT(*) FILTER (WHERE Herbicides = true)
                OVER (PARTITION BY departement_code, culture) AS nb_periodes_herbicides,
            COUNT(*) FILTER (WHERE Fongicides = true OR Insecticides = true)
                OVER (PARTITION BY departement_code, culture) AS nb_periodes_fongi_insecti,
            COUNT(*) OVER (PARTITION BY departement_code, culture) AS nb_periodes_total
        FROM calendrier_epandage
    """).pl()

    meteo = None
    if not METEO_ENABLED:
        print("METEO_ENABLED=False : chargement météo ignoré, indicateur_meteo=1 utilisé par défaut")
    else:
        try:
            meteo = pl.read_parquet(
                PARQUET_DIR / f"meteo/historique/{annee}/meteo.parquet"
            ).select(["code_insee", "time", "wind_speed_10m_max", "precipitation_sum"])
        except FileNotFoundError:
            print(f"Fichier météo introuvable pour l'année {annee} -> mode sans météo")
            meteo = None

    con.close()
    return ift, cal, meteo

In [16]:
# Chargement des données
ift_data, cal_data, meteo = load_data(ANNEE, METEO_ENABLED=True)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [13]:
# Verification
print("Calendrier Epandage")
display(cal_data)
print("Données IFT")
display(ift_data)

Calendrier Epandage


departement_code,culture,Debut_de_periode,Fin_de_periode,Herbicides,Fongicides,Insecticides,nb_periodes_herbicides,nb_periodes_fongi_insecti,nb_periodes_total
i64,str,date,date,bool,bool,bool,i64,i64,i64
18,"""Orge d'hiver""",2025-01-01,2025-03-14,true,false,false,4,4,6
18,"""Orge d'hiver""",2025-03-15,2025-04-30,true,true,false,4,4,6
18,"""Orge d'hiver""",2025-05-01,2025-05-14,false,true,true,4,4,6
18,"""Orge d'hiver""",2025-05-15,2025-06-30,false,true,true,4,4,6
18,"""Orge d'hiver""",2025-12-01,2025-12-31,true,false,false,4,4,6
…,…,…,…,…,…,…,…,…,…
41,"""Vigne""",2025-02-15,2025-04-14,true,false,false,5,7,10
41,"""Vigne""",2025-04-15,2025-05-14,false,true,false,5,7,10
41,"""Vigne""",2025-07-15,2025-07-31,false,false,true,5,7,10


Données IFT


insee_com,code_insee_dep,c_maj,ift_maj_hbc,c_ift_hbc,ift_hh_hbc,c_ift_h,ift_h,c_maj_cal,c_ift_hbc_cal,c_ift_h_cal
str,str,str,f64,str,f64,str,f64,str,str,str
"""01001""","""01""","""Maïs""",2.25,"""Blé tendre""",1.27,"""Maïs""",0.99,"""Maïs""","""Blé tendre""","""Maïs"""
"""01002""","""01""","""Prairie permanente""",0.39,"""Vigne""",0.37,"""Vigne""",0.02,"""Prairies""","""Vigne""","""Vigne"""
"""01004""","""01""","""Prairie permanente""",0.56,"""Maïs""",0.3,"""Maïs""",0.26,"""Prairies""","""Maïs""","""Maïs"""
"""01005""","""01""","""Maïs""",2.26,"""Blé tendre""",1.21,"""Maïs""",1.05,"""Maïs""","""Blé tendre""","""Maïs"""
"""01006""","""01""","""Prairie permanente""",0.21,"""Blé tendre""",0.12,"""Blé tendre""",0.09,"""Prairies""","""Blé tendre""","""Blé tendre"""
…,…,…,…,…,…,…,…,…,…,…
"""97613""","""97""","""Autre culture non précisée dan…",0.52,"""Banane""",0.44,"""Banane""",0.08,null,null,null
"""97614""","""97""","""Autre culture non précisée dan…",0.45,"""Banane""",0.39,"""Banane""",0.06,null,null,null
"""97615""","""97""","""Autre culture non précisée dan…",0.0,"""Aucune""",0.0,"""Aucune""",0.0,null,null,null


In [14]:
# Simuler jour par jour pour la commune
nb_jours = 366 if ANNEE % 4 == 0 else 365
dates    = [date(ANNEE, 1, 1) + timedelta(days=d) for d in range(nb_jours)]

resultats = []
for d in dates:
    ift_j = compute_ift_journalier(ift_data, cal_data, d)
    row_j = ift_j.filter(pl.col('insee_com') == INSEE)
    if len(row_j) > 0:
        val = row_j['ift_journalier_total'][0]
        ift_val = float(val) if val is not None else float('nan')
    else:
        ift_val = float('nan')
    resultats.append({'date': d, 'ift_journalier': ift_val})

ts = pd.DataFrame(resultats)
print(f"IFT journalier max        : {ts['ift_journalier'].max():.4f}")
print(f"Jours avec IFT > 0        : {(ts['ift_journalier'] > 0).sum()}")
print(f"Jours sans données (NaN)  : {ts['ift_journalier'].isna().sum()}")
print(f"Somme IFT annuel reconstituée : {ts['ift_journalier'].sum():.3f}")
print(f"IFT annuel source (ift_t_hbc) : {r.get('ift_maj_hbc', 0):.3f}")

IFT journalier max        : 1.2675
Jours avec IFT > 0        : 304
Jours sans données (NaN)  : 0
Somme IFT annuel reconstituée : 327.910
IFT annuel source (ift_t_hbc) : 0.000


In [15]:
ts

,date,ift_journalier
0,2025-01-01,1.0300
1,2025-01-02,1.0300
2,2025-01-03,1.0300
3,2025-01-04,1.0300
4,2025-01-05,1.0300
...,...,...
360,2025-12-27,1.2675
361,2025-12-28,1.2675
362,2025-12-29,1.2675
363,2025-12-30,1.2675


## 5. Visualisation

In [16]:
# Palette risque
RISK_COLORS = {0: '#A5D6A7', 1: '#FFF176', 2: '#FFB300', 3: '#F57C00', 4: '#B71C1C'}

# Normalisation 0-4 simplifiée (quartiles)
vals_pos = ts[ts['ift_journalier'] > 0]['ift_journalier']
if len(vals_pos) > 0:
    if METHODE_SEUIL == "quartiles":
        q1, q2, q3 = vals_pos.quantile([0.25, 0.5, 0.75])
    else:
        q1, q2, q3 = VALEURS_SEUIL  # Seuils fixes pour les niveaux de risque
    print("Valeurs seuils: ", [q1, q2, q3])
    def niveau(v):
        import math
        if math.isnan(v): return float('nan')  # pas de données calendrier
        if v == 0: return 0
        if v <= q1: return 1
        if v <= q2: return 2
        if v <= q3: return 3
        return 4
    ts['risque_0_4'] = ts['ift_journalier'].apply(niveau)
else:
    ts['risque_0_4'] = 0
ts['color'] = ts['risque_0_4'].map(RISK_COLORS)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4],
    subplot_titles=[
        f'IFT journalier — {r.get("nom_commune", INSEE)} ({INSEE})',
        'Indicateur de risque 0-4'
    ],
    vertical_spacing=0.08
)

# IFT journalier
fig.add_trace(go.Bar(
    x=ts['date'], y=ts['ift_journalier'],
    marker_color=ts['color'],
    hovertemplate='<b>%{x|%d/%m/%Y}</b><br>IFT : %{y:.4f}<extra></extra>',
    name='IFT journalier'
), row=1, col=1)

# Périodes de traitement du calendrier (vigne dep 37)
colors_type = {'Herbicides': 'rgba(255,100,0,0.15)', 'Tout': 'rgba(0,100,255,0.1)'}
for _, p in cal_commune[cal_commune['culture'].isin(cultures_commune)].iterrows():
    label_p = 'Herbicides' if p['Herbicides'] else 'Tout'
    fig.add_vrect(
        x0=str(p['debut']), x1=str(p['fin']),
        fillcolor=colors_type.get(label_p, 'rgba(128,128,128,0.1)'),
        opacity=1, line_width=0,
        annotation_text=f"{p['culture'][:4]} {label_p[:4]}",
        annotation_font_size=12,
        row=1, col=1
    )

# Risque 0-4
fig.add_trace(go.Bar(
    x=ts['date'], y=ts['risque_0_4'],
    marker_color=ts['color'],
    hovertemplate='<b>%{x|%d/%m/%Y}</b><br>Risque : %{y}<extra></extra>',
    name='Risque 0-4', showlegend=False
), row=2, col=1)

fig.update_layout(
    height=600, bargap=0,
    hovermode='x unified',
    margin=dict(l=50, r=20, t=50, b=20),
    showlegend=False
)
fig.update_yaxes(title_text='IFT/jour', row=1)
fig.update_yaxes(title_text='Risque', tickvals=[0,1,2,3,4], range=[0,4.3], row=2)
fig.show()

Valeurs seuils:  [1.03, 1.03, 1.2675]


## 6. Détail par période de traitement

In [18]:
# Pour chaque période du calendrier, afficher l'IFT distribué et la cohérence
print(f"{'Culture':15s} {'Période':25s} {'Herb':5s} {'Fongi':5s} {'Insec':5s} {'IFT/j (total)':>14} {'IFT/j (h)':>10} {'IFT/j (f+i)':>12}")
print('-' * 100)

for _, p in cal_commune[cal_commune['culture'].isin(cultures_commune)].sort_values(['culture','debut']).iterrows():
    # IFT journalier prévu pour une date dans cette période
    test_date = p['debut'] + timedelta(days=1)
    ift_j = compute_ift_journalier(ift_data, cal_data, test_date)
    row_j  = ift_j.filter(pl.col('insee_com') == INSEE)
    val = row_j['ift_journalier_total'][0] if len(row_j) > 0 else None
    ift_val = float(val) if val is not None else float('nan')
    
    print(
        f"  {p['culture']:13s} "
        f"{str(p['debut'])} → {str(p['fin'])}  "
        f"{'✓' if p['Herbicides'] else ' ':5s}"
        f"{'✓' if p['Fongicides'] else ' ':5s}"
        f"{'✓' if p['Insecticides'] else ' ':5s}"
        f"  {ift_val:>12.4f}"
    )

Culture         Période                   Herb  Fongi Insec  IFT/j (total)  IFT/j (h)  IFT/j (f+i)
----------------------------------------------------------------------------------------------------
  Blé tendre    2025-02-01 00:00:00 → 2025-03-14 00:00:00  ✓                      1.2675
  Blé tendre    2025-03-15 00:00:00 → 2025-04-14 00:00:00       ✓                 1.0300
  Blé tendre    2025-04-15 00:00:00 → 2025-04-30 00:00:00  ✓    ✓                 1.2675
  Blé tendre    2025-05-01 00:00:00 → 2025-06-14 00:00:00       ✓    ✓            1.0300
  Blé tendre    2025-10-01 00:00:00 → 2025-10-14 00:00:00            ✓            0.0000
  Blé tendre    2025-10-15 00:00:00 → 2025-11-30 00:00:00  ✓         ✓            0.2375
  Blé tendre    2025-12-01 00:00:00 → 2025-12-31 00:00:00  ✓                      1.2675
  Vigne         2025-01-01 00:00:00 → 2025-03-30 00:00:00  ✓                      1.0300
  Vigne         2025-04-01 00:00:00 → 2025-04-30 00:00:00  ✓    ✓                 1.0300